# ToysAD-8K Development Identity Evaluator Release

**Notebook file:** `08e1-toysad8k-development-identity-evaluator-release.ipynb`

This CPU stage opens reference object masks only after all development predictions are locked. It maps each predicted object to an opaque reference identity for Notebook 08F. Only the evaluator resource is released. No evaluator field enters the model-facing path.

## Acceptance contract

- The fixed cohort contains exactly 1,000 scenes and 20,000 RGB views.
- Cohort membership and prerequisite hashes must match before source bytes are read.
- Bounded incremental runs save deterministic per-scene state and a verified recovery ZIP.
- A final resource manifest is written only after exact full coverage.
- Reference boxes use the established grayscale mask threshold of 128 and half-open XYXY coordinates.
- ClearML receives aggregate recovery or final evidence only.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import subprocess
import sys

sys.dont_write_bytecode = True
from IPython.display import display

EXECUTION_ENVIRONMENT = "kaggle" if Path("/kaggle/input").is_dir() else "local"
OUTPUT_ROOT = Path(os.getenv(
    "VPI_OUTPUT_ROOT",
    "/kaggle/working/08e1_toysad8k_development_identity_evaluator_release" if EXECUTION_ENVIRONMENT == "kaggle"
    else ".local/runs/08e1_toysad8k_development_identity_evaluator_release",
)).expanduser().resolve()
RUN_MODE = os.getenv("VPI_RUN_MODE", "full")
MAXIMUM_NEW_SCENES = int(os.getenv("VPI_MAXIMUM_NEW_SCENES", "100"))

In [ ]:
def kaggle_roots() -> list[Path]:
    """Return dataset and notebook mounts without their container directories."""
    root = Path("/kaggle/input")
    if not root.is_dir():
        return []
    candidates = [path for path in root.iterdir()
                  if path.is_dir() and path.name not in {"datasets", "notebooks", "models"}]
    for pattern in ("datasets/*/*", "notebooks/*/*", "models/*/*/*/*"):
        for path in root.glob(pattern):
            if not path.is_dir():
                continue
            versions = path / "versions"
            if versions.is_dir():
                candidates.extend(child for child in versions.iterdir() if child.is_dir())
            else:
                candidates.append(path)
    return sorted({path.resolve() for path in candidates}, key=lambda path: path.as_posix())


def find_code_root() -> Path:
    configured = os.getenv("VPI_CODE_ROOT")
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend((Path.cwd(), *Path.cwd().parents, *kaggle_roots()))
    for candidate in candidates:
        for value in (candidate, candidate / "code"):
            if (value / "scripts/run_primary_resource_gate.py").is_file():
                return value.resolve()
    raise FileNotFoundError("The Visual Peer Inspector code snapshot was not found")


def find_directory(marker: str, environment_name: str) -> Path:
    configured = os.getenv(environment_name)
    if configured:
        candidate = Path(configured).expanduser()
        if (candidate / marker).is_file():
            return candidate.resolve()
    matches = []
    for root in kaggle_roots():
        candidates = (root, *(path for path in root.iterdir() if path.is_dir()))
        matches.extend(candidate.resolve() for candidate in candidates if (candidate / marker).is_file())
    unique = sorted(set(matches), key=lambda path: path.as_posix())
    if len(unique) != 1:
        raise FileNotFoundError(f"Expected one mounted directory containing {marker}; found {len(unique)}")
    return unique[0]


def mounted_files(name: str) -> list[Path]:
    """Find small release files without entering raw images or saved checkpoints."""
    matches = set()
    excluded = {"media", "toysAD8K", "toysad8k", "trusted-release-state", "checkpoints", ".git"}
    for root in kaggle_roots():
        # The full source mount has over a million files and no prediction locks.
        if (root / "archive_index.sqlite3").is_file():
            continue
        pending = [(root, 0)]
        while pending:
            directory, depth = pending.pop()
            candidate = directory / name
            if candidate.is_file() and not candidate.is_symlink():
                matches.add(candidate.resolve())
            if depth == 5:
                continue
            for child in directory.iterdir():
                if child.name in excluded or child.is_symlink() or not child.is_dir():
                    continue
                if (child / "archive_index.sqlite3").is_file():
                    continue
                pending.append((child, depth + 1))
    return sorted(matches, key=lambda path: path.as_posix())


def find_named_file(name: str, environment_name: str) -> Path:
    configured = os.getenv(environment_name)
    if configured and Path(configured).expanduser().is_file():
        return Path(configured).expanduser().resolve()
    files = mounted_files(name)
    if len(files) != 1:
        raise FileNotFoundError(f"Expected one mounted {name}; found {len(files)}")
    return files[0]


def find_prediction_roots() -> list[Path]:
    configured = os.getenv("VPI_PREDICTION_ROOTS")
    if configured:
        return [Path(value).expanduser().resolve() for value in configured.split(os.pathsep)]
    expected = {
        "toysad8k-development-screen-100-v1",
        "toysad8k-development-confirm-300-v1",
        "toysad8k-development-remainder-600-v1",
    }
    roots = []
    for lock_path in mounted_files("prediction_lock.json"):
        value = json.loads(lock_path.read_text(encoding="utf-8"))
        if value.get("cohort_id") in expected:
            roots.append(lock_path.parent.resolve())
    if {json.loads((root / "prediction_lock.json").read_text())["cohort_id"] for root in roots} != expected:
        raise FileNotFoundError("The three completed Notebook 08E prediction roots were not found")
    return sorted(set(roots), key=lambda path: path.as_posix())


def find_recovery_pair(mode: str) -> tuple[Path, Path] | None:
    stem = f"{mode}-resource-recovery"
    bundles = mounted_files(f"{stem}.zip") + mounted_files(f"{stem}.zip.bin")
    manifests = mounted_files(f"{stem}-manifest.json")
    if not bundles and not manifests:
        return None
    if len(bundles) != 1 or len(manifests) != 1:
        raise ValueError("Attach exactly one prior recovery output version")
    return bundles[0], manifests[0]


def code_environment() -> dict[str, str]:
    environment = os.environ.copy()
    source = str(CODE_ROOT / "src")
    current = environment.get("PYTHONPATH")
    environment["PYTHONPATH"] = source if not current else source + os.pathsep + current
    return environment


def clearml_mode() -> str:
    if EXECUTION_ENVIRONMENT != "kaggle":
        return "disabled"
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        access = client.get_secret("CLEARML_API_ACCESS_KEY")
        secret = client.get_secret("CLEARML_API_SECRET_KEY")
    except Exception:
        return "disabled"
    if not access or not secret:
        return "disabled"
    os.environ.update({
        "CLEARML_API_ACCESS_KEY": access,
        "CLEARML_API_SECRET_KEY": secret,
        "CLEARML_API_HOST": "https://api.clear.ml",
        "CLEARML_WEB_HOST": "https://app.clear.ml",
        "CLEARML_FILES_HOST": "https://files.clear.ml",
    })
    return "online"

## Resolve and verify inputs

Inputs are located by exact marker files. No broad archive extraction is used.

In [ ]:
CODE_ROOT = find_code_root()
sys.path.insert(0, str(CODE_ROOT / "src"))
from visual_peer_inspector.notebook_provenance import verify_notebook_code_snapshot

CODE_PROVENANCE = verify_notebook_code_snapshot(
    CODE_ROOT,
    "notebooks/development-freeze/08e1-toysad8k-development-identity-evaluator-release.ipynb",
)
print("Code snapshot verified", flush=True)
SOURCE_ROOT = find_directory("archive_index.sqlite3", "VPI_SOURCE_ROOT")
ADMINISTRATIVE_ROOT = find_directory("restricted_source_map.jsonl", "VPI_ADMINISTRATIVE_ROOT")
print("Source and selection mounts found", flush=True)
RECOVERY_DIRECTORY = OUTPUT_ROOT.parent / (OUTPUT_ROOT.name + "_recovery")
command = [
    sys.executable,
    str(CODE_ROOT / "scripts/run_primary_resource_gate.py"),
    "--mode", "development-identity",
    "--administrative-root", str(ADMINISTRATIVE_ROOT),
    "--source-root", str(SOURCE_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--run-mode", RUN_MODE,
    "--maximum-new-scenes", str(MAXIMUM_NEW_SCENES),
    "--recovery-output-directory", str(RECOVERY_DIRECTORY),
]

for prediction_root in find_prediction_roots():
    command.extend(("--prediction-root", str(prediction_root)))

recovery = find_recovery_pair("development-identity")
print("Release inputs located; starting verified resource build", flush=True)
if recovery is not None:
    command.extend(("--recovery-bundle", str(recovery[0]), "--recovery-manifest", str(recovery[1])))
subprocess.run(command, check=True, env=code_environment())
result = json.loads((OUTPUT_ROOT / "resource_run_result.json").read_text(encoding="utf-8"))
display({
    "status": result["status"],
    "completed scenes": result["completed_scene_count"],
    "remaining scenes": result["remaining_scene_count"],
    "processed this version": result["processed_scene_count"],
    "resumed scenes": result["resumed_scene_count"],
    "exact resume": result["exact_resume"],
})

## Release state

A `complete` result is ready for private dataset publication. A `checkpointed` result publishes only the recovery pair, which can be attached to the next Kaggle version.

In [ ]:
display({key: value for key, value in result.items() if key not in {'recovery'}})
if 'recovery' in result:
    display(result['recovery'])

## Bounded ClearML finalisation

Tracking starts after local evidence exists and cannot change the release result.

In [ ]:
SAFE_ROOT = OUTPUT_ROOT / ("safe" if result["status"] == "complete" else "tracking-safe")
mode = clearml_mode()
receipt = OUTPUT_ROOT / "tracking" / "clearml_receipt.json"
if mode == "online":
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
             "--retries", "2", "--timeout", "30",
             "-r", str(CODE_ROOT / "requirements/confirmation-tracking-lock.txt")],
            check=True, timeout=180, env=code_environment(),
        )
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        mode = "disabled"
        display("Tracking dependencies were unavailable. The saved release can be published separately.")
try:
    subprocess.run(
        [
            sys.executable,
            str(CODE_ROOT / "scripts/finalize_ds2_clearml.py"),
            "--safe-root", str(SAFE_ROOT),
            "--receipt", str(receipt),
            "--stage", "development_identity_evaluator_release",
            "--project", "Visual Peer Inspector/05 Development Freeze",
            "--task-name", "08e1-primary-resource-gate",
            "--tag", "notebook-08e1",
            "--tag", "final" if result["status"] == "complete" else "recovery",
            "--mode", mode,
            "--wall-time-seconds", "180",
        ],
        check=False,
        timeout=240,
        env=code_environment(),
    )
except subprocess.TimeoutExpired:
    display("ClearML timed out after the scientific output was saved")
display({"ClearML mode": mode, "receipt": str(receipt)})
if receipt.is_file():
    display(json.loads(receipt.read_text(encoding="utf-8")))

## Handoff

Publish the completed output as the private Kaggle dataset `vpi-toysad-development-identity-evaluator`. Model input and evaluator output must remain separate. If the stage is incomplete, attach the recovery ZIP and manifest to a new version and run again. Do not publish trusted release state, credentials, or evaluator data as model input.